# 01 — Data Validation & Schema Audit
## EMIPredict AI Platform
This notebook performs comprehensive schema verification, data quality checks, null/inf detection, and domain boundary audits on `data/raw/EMI_dataset.csv` using modular functions from `src.data`.


In [ ]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.data.load_data import load_raw_dataset
from src.data.validate_data import run_data_validation


### 1. Load Raw Dataset with Memory Downcasting
We load the raw CSV dataset, downcast floating and integer columns to reduce RAM footprint, and verify dataset dimensions.


In [ ]:
df = load_raw_dataset()
print(f'Dataset Shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB')
df.head()


### 2. Execute Full Data Quality Audit
We execute the automated data validation suite, checking for:
- Duplicated columns and rows
- Missing values per column
- Infinite numerical values
- Constant & near-constant columns
- Categorical anomalies & target distribution
- Target leakage risks


In [ ]:
validation_report = run_data_validation(df)
print('Data Validation Audit Completed.')
print(f'Total Rows: {validation_report["total_rows"]:,}')
print(f'Exact Duplicate Rows: {validation_report["duplicate_rows"]["count"]:,}')
print(f'Missing Cells: {validation_report["missing_values_summary"]["total_missing_cells"]}')


### 3. Target Distribution Verification
We check the balance of our multiclass target `emi_eligibility` and key statistical moments of `max_monthly_emi`.


In [ ]:
print('Classification Target Distribution:')
print(df['emi_eligibility'].value_counts())

print('\nRegression Target Summary (₹):')
print(df['max_monthly_emi'].describe())


### Summary & Next Steps
The raw dataset contains 404,800 records across 25 input variables and 2 target variables. All records have valid types, and the validation report has been saved to `reports/data_quality_report.json`.
